# Pipeline ELT — ITBI Recife

**Fluxo:**
1. **Extract** — leitura do CSV unificado (dados brutos)
2. **Load** — carga em `staging.itbi_raw` sem nenhum tratamento
3. **Transform** — executado pelo **dbt** (`dbt run` no terminal)

## 1. Instalação de dependências

In [3]:
%pip install psycopg2-binary sqlalchemy pandas python-dotenv --quiet


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Conexão com o banco (Supabase PostgreSQL)

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '.env'), override=True)

usuario_db    = os.getenv('DB_USER')
senha_db      = os.getenv('DB_PASSWORD')
host_db       = os.getenv('DB_HOST')
porta_db      = int(os.getenv('DB_PORT'))
nome_banco_db = os.getenv('DB_NAME')

faltando = [k for k, v in {'DB_USER': usuario_db, 'DB_PASSWORD': senha_db, 'DB_HOST': host_db, 'DB_PORT': porta_db, 'DB_NAME': nome_banco_db}.items() if not v]
if faltando:
    raise ValueError(f'Variáveis ausentes no .env: {faltando}')

engine = create_engine(URL.create(
    drivername="postgresql+psycopg2",
    username=usuario_db,
    password=senha_db,
    host=host_db,
    port=porta_db,
    database=nome_banco_db,
    query={"client_encoding": "utf8"}
))

with engine.connect() as conn:
    print("Conectado:", conn.execute(text("SELECT version()")).scalar()[:50])

## 4. Extract — leitura do CSV

In [10]:
df = pd.read_csv("../Planilhas/imoveis_unificado.csv", sep=";", dtype=str, low_memory=False)

print(f"Linhas lidas : {len(df):,}")
print(f"Colunas      : {list(df.columns)}")
df.head(3)

Linhas lidas : 44,232
Colunas      : ['logradouro', 'numero', 'complemento', 'valor_avaliacao', 'bairro', 'cidade', 'uf', 'ano_construcao', 'area_terreno', 'area_construida', 'fracao_ideal', 'padrao_acabamento', 'tipo_construcao', 'tipo_ocupacao', 'data_transacao', 'estado_conservacao', 'tipo_imovel', 'sfh', 'cod_logradouro', 'latitude', 'longitude', 'ano', 'distrito']


,logradouro,numero,complemento,valor_avaliacao,bairro,cidade,uf,ano_construcao,area_terreno,area_construida,...,tipo_ocupacao,data_transacao,estado_conservacao,tipo_imovel,sfh,cod_logradouro,latitude,longitude,ano,distrito
0,logradouro,numero,complemento,valor_avaliacao,bairro,cidade,uf,ano_construcao,area_terreno,area_construida,...,tipo_ocupacao,data_transacao,estado_conservacao,tipo_imovel,sfh,cod_logradouro,latitude,longitude,ano,Distrito
1,av norte miguel arraes de alencar,3071,NaN,1068562.63,Encruzilhada,Recife,PE,1997,"438,0","511,0",...,COMERCIAL COM LIXO ORGANICO,2023-12-21,Regular,Galpão,0.00,46540,-8.034273,-34.896337,2023,1
2,av norte miguel arraes de alencar,3029,NaN,1500000.00,Encruzilhada,Recife,PE,1957,"779,33","582,44",...,COMERCIAL SEM LIXO ORGANICO,2023-11-17,Regular,Casa,0.00,46540,-8.034435,-34.896335,2023,1


## 5. Load — carga bruta em `staging.itbi_raw`

Nenhum tratamento é aplicado aqui. Os dados entram no banco exatamente como estão no CSV.

In [12]:
with engine.connect() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS staging"))
    conn.commit()

df.to_sql(
    name="itbi_raw",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False,
    chunksize=5000
)

with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM staging.itbi_raw")).scalar()

print(f"staging.itbi_raw: {total:,} linhas carregadas")

staging.itbi_raw: 44,232 linhas carregadas


## 3. Validação — staging populado

Confirma que o Load foi bem-sucedido antes de rodar o dbt.

In [13]:
with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM staging.itbi_raw")).scalar()
    print(f"staging.itbi_raw: {total:,} linhas")
    print("Pronto. Rode agora: cd ../itbi_dw && dbt run")

staging.itbi_raw: 44,232 linhas
Pronto. Rode agora: cd ../itbi_dw && dbt run


## 4. Análises e Insights

As queries abaixo são executadas após o `dbt run`, lendo o `dw_elt` já populado.

In [ ]:
# Análise 1 — Top 10 bairros por valor médio de avaliação
sql = """
SELECT
    l.bairro,
    COUNT(*)                             AS transacoes,
    ROUND(AVG(f.valor_avaliacao), 2)     AS valor_medio,
    ROUND(AVG(f.valor_m2_construido), 2) AS valor_medio_m2
FROM dw_elt.fato_transacao_itbi f
JOIN dw_elt.dim_localizacao l ON l.sk_localizacao = f.sk_localizacao
WHERE f.valor_avaliacao IS NOT NULL
GROUP BY l.bairro
ORDER BY valor_medio DESC
LIMIT 10
"""

with engine.connect() as conn:
    df_analise1 = pd.read_sql(text(sql), conn)

print("Top 10 bairros por valor médio de avaliação:")
df_analise1

In [ ]:
# Análise 2 — Evolução anual e trimestral do volume de transações
sql = """
SELECT
    t.ano,
    t.trimestre,
    COUNT(*)                         AS transacoes,
    ROUND(AVG(f.valor_avaliacao), 2) AS valor_medio_avaliacao
FROM dw_elt.fato_transacao_itbi f
JOIN dw_elt.dim_tempo t ON t.sk_tempo = f.sk_tempo
GROUP BY t.ano, t.trimestre
ORDER BY t.ano, t.trimestre
"""

with engine.connect() as conn:
    df_analise2 = pd.read_sql(text(sql), conn)

print("Evolução trimestral do volume de transações:")
df_analise2

In [ ]:
# Análise 3 — Valor médio por padrão de acabamento e tipo de imóvel
sql = """
SELECT
    c.tipo_imovel,
    c.padrao_acabamento,
    COUNT(*)                             AS transacoes,
    ROUND(AVG(f.valor_avaliacao), 2)     AS valor_medio,
    ROUND(AVG(f.valor_m2_construido), 2) AS valor_medio_m2
FROM dw_elt.fato_transacao_itbi f
JOIN dw_elt.dim_caracteristica c ON c.sk_caracteristica = f.sk_caracteristica
WHERE f.valor_m2_construido IS NOT NULL
GROUP BY c.tipo_imovel, c.padrao_acabamento
ORDER BY c.tipo_imovel, valor_medio_m2 DESC
"""

with engine.connect() as conn:
    df_analise3 = pd.read_sql(text(sql), conn)

print("Valor médio por padrão de acabamento e tipo de imóvel:")
df_analise3